# Zero-DCE - Phase B: Decoupling Noise Gain from Contrast Gain

**Status: not yet run.** The SESSION START cell is ready. Experiment 5.0 is the
gate; 5.1-5.3 only mean something once 5.0 passes.

## The idea in one line

A pointwise curve multiplies detail and noise by the *same* number, so it cannot
change signal-to-noise ratio. Split the image into a smooth **base** and a
**detail** layer, brighten the base with the curve, and give the detail its own
gain. Brightness and noise then have separate knobs.

## The modified recurrence

```
b = blur(x)            base
d = x - b              detail (fine texture AND noise)

x_n = f_n(b + gamma*d) + (1 - gamma)*d
```

`f_n` is the **unmodified** Zero-DCE curve. Expanding:

```
x_n = f_n(b) + [gamma*g_b + (1 - gamma)]*d + gamma^2*A_n*d^2
```

| gamma | detail gain | meaning |
|---|---|---|
| 1 | `g_b` | bit-exact vanilla Zero-DCE |
| 0 | 1 | full brightening, noise never amplified |

Vanilla Zero-DCE is the special case where the two gains are forced equal, so
this is a strict generalisation, not a replacement.

## Why bother

Phase A measured 76.99x cumulative variance amplification. That is 8.77x in
standard deviation - which is also the factor by which dark-region contrast is
amplified. They are the same number. Penalising it (the old `W_noise` work)
makes the model dim the image, which is exactly what happened: PSNR 17.26 -> 14.74
at W50, -> 9.21 at W400.

Decoupling requires leaving pointwise operation. That is the whole contribution.

## Cost

Blur per iteration is roughly 1% of 85.5 GFLOPs; extra output channels add low
single digits. Call it "about Zero-DCE cost", not zero. Measure with `thop`
rather than estimating, and report wall-clock too - blurs are memory-bound, so
timing looks worse than FLOPs suggest. Compare against ZLEN: 3.7 ms -> 5.2 ms (+40%).

## SESSION START

Run first after every runtime restart. Must print `PHASE B READY`.

In [ ]:
# ============================================================
# PHASE B - SESSION START
# Decoupling noise gain from contrast gain in Zero-DCE.
# No training. Baseline checkpoint only.
# Run this first after every runtime restart.
# ============================================================

import os, sys, json, hashlib, subprocess
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from pathlib import Path
from PIL import Image
from datetime import datetime
from tqdm.auto import tqdm

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "scikit-image", "pandas"], check=False)

# ---- repository: one pull ----
REPO_DIR = "/content/Zero-DCE"
BRANCH   = "claude/nifty-ritchie-qpqakn"
CODE_DIR = f"{REPO_DIR}/Zero-DCE_code"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone",
                    "https://github.com/shreyaio/Zero-DCE.git", REPO_DIR], check=True)

def git_run(a):
    return subprocess.check_output(["git"] + a, cwd=REPO_DIR, text=True).strip()

git_run(["fetch", "origin", BRANCH])
git_run(["checkout", BRANCH])
git_run(["pull", "--ff-only", "origin", BRANCH])

current_branch, commit = git_run(["branch", "--show-current"]), git_run(["rev-parse", "HEAD"])
print("Branch:", current_branch, "| Commit:", commit)

os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# ---- device: both spellings ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE = device
print("Device:", device,
      "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")

# ---- dataset ----
LOL_ROOT = "/content/lol_dataset/lol_dataset/eval15"
if not os.path.isdir(f"{LOL_ROOT}/low"):
    from huggingface_hub import hf_hub_download
    import shutil
    os.makedirs("/content/lol_dataset", exist_ok=True)
    shutil.unpack_archive(
        hf_hub_download(repo_id="geekyrakshit/LoL-Dataset",
                        filename="lol_dataset.zip", repo_type="dataset"),
        "/content/lol_dataset")

LOW_DIR, HIGH_DIR = Path(f"{LOL_ROOT}/low"), Path(f"{LOL_ROOT}/high")
assert LOW_DIR.is_dir() and HIGH_DIR.is_dir()

image_paths = sorted(p for p in LOW_DIR.iterdir()
                     if p.suffix.lower() in [".png", ".jpg", ".jpeg", ".bmp"])
print("Images:", len(image_paths))

# ---- configuration ----
BRIGHTNESS_BINS = {"dark": (0.00, 0.10), "mid_dark": (0.10, 0.25),
                   "mid_bright": (0.25, 0.50), "bright": (0.50, 1.01)}
SIGMAS, N_TRIALS, BASE_SEED = [0.005, 0.010, 0.020], 30, 42026

PHASE_B_DIR = Path("/content/drive/MyDrive/zero-dce/results/phase_b")
PHASE_B_DIR.mkdir(parents=True, exist_ok=True)

# ---- model ----
import model as model_module

CKPT = "/content/drive/MyDrive/zero-dce/checkpoints/Epoch199.pth"
assert os.path.isfile(CKPT), f"Checkpoint missing: {CKPT}"

model = model_module.enhance_net_nopool().to(device)
model.load_state_dict(torch.load(CKPT, map_location=device))
model.eval()
net = model

params = sum(p.numel() for p in model.parameters())
print("Parameters:", params)
assert params == 79416, f"Expected 79416, got {params}"

# ---- helpers ----
def load_rgb_image(path):
    a = np.asarray(Image.open(path).convert("RGB")).astype(np.float32) / 255.0
    return torch.from_numpy(a).permute(2, 0, 1).unsqueeze(0).to(device)


@torch.no_grad()
def apply_zero_dce_curves(image, curves):
    b, _, h, w = image.shape
    steps = curves.reshape(b, 8, 3, h, w)
    x = image.clone()
    for i in range(8):
        A = steps[:, i]
        x = x + A * (x * x - x)
    return x


@torch.no_grad()
def calculate_fixed_curve_jacobian(clean, curves):
    b, _, h, w = clean.shape
    steps = curves.reshape(b, 8, 3, h, w)
    x, J = clean.clone(), torch.ones_like(clean)
    for i in range(8):
        A = steps[:, i]
        J = J * (1.0 + A * (2.0 * x - 1.0))
        x = x + A * (x * x - x)
    return J


# ---- smoke test ----
_probe = load_rgb_image(image_paths[0])
with torch.no_grad():
    _, _enh, _curves = model(_probe)
_err = (apply_zero_dce_curves(_probe, _curves) - _enh).abs().max().item()

print(f"\nRecurrence smoke test - max abs error: {_err:.3e}")
print("PHASE B READY" if _err < 1e-5 else "FAILED - stop here")

## Experiment 5.0 - decomposition and exactness gate

**Hard gate.** `gamma = 1` must reproduce vanilla Zero-DCE.

Expected `max_abs_error` around `1e-7` (float32 round-off). Above `1e-5` means
the decomposition is wrong - stop and fix the blur radius before going on.

Also reports how much image energy lives in the detail layer, per brightness
region. That tells us how much a per-pixel gamma would have to work with.

In [ ]:
# ============================================================
# EXPERIMENT 5.0 - decomposition and exactness gate
#
# x_n = f_n(b + gamma*d) + (1 - gamma)*d,   b = blur(x), d = x - b
#
# gamma = 1  ->  bit-exact vanilla Zero-DCE
# gamma = 0  ->  full brightening, detail gain exactly 1
#
# GATE: gamma = 1 must reproduce vanilla. Stop if it does not.
# ============================================================

EXP5_DIR = PHASE_B_DIR / "experiment_5"
EXP5_DIR.mkdir(parents=True, exist_ok=True)

BASE_BLUR_SIGMA = 1.5          # pixels
GAMMAS = [1.0, 0.7, 0.5, 0.3, 0.0]
EXP5_SEED = 52026

model.eval()


def make_gaussian_kernel(sigma, device, dtype=torch.float32):
    radius = max(1, int(round(3.0 * sigma)))
    coords = torch.arange(-radius, radius + 1, device=device, dtype=dtype)
    kernel = torch.exp(-(coords ** 2) / (2.0 * sigma * sigma))
    return kernel / kernel.sum(), radius


def gaussian_base(image, kernel, radius):
    """Separable Gaussian blur, reflect padding. Preserves the mean."""
    channels = image.shape[1]
    kx = kernel.view(1, 1, 1, -1).expand(channels, 1, 1, -1)
    ky = kernel.view(1, 1, -1, 1).expand(channels, 1, -1, 1)
    y = F.pad(image, (radius, radius, 0, 0), mode="reflect")
    y = F.conv2d(y, kx, groups=channels)
    y = F.pad(y, (0, 0, radius, radius), mode="reflect")
    y = F.conv2d(y, ky, groups=channels)
    return y


@torch.no_grad()
def gamma_recurrence(image, curves, gamma, kernel, radius, return_stages=False):
    """gamma may be a float or a per-pixel tensor broadcastable to the image."""
    batch, _, height, width = image.shape
    curve_steps = curves.reshape(batch, 8, 3, height, width)

    x = image.clone()
    stages = [x.clone()] if return_stages else None

    for iteration in range(8):
        A = curve_steps[:, iteration]

        base = gaussian_base(x, kernel, radius)
        detail = x - base

        y = base + gamma * detail
        y = y + A * (y * y - y)             # unmodified Zero-DCE curve
        x = y + (1.0 - gamma) * detail      # re-inject untouched remainder

        if return_stages:
            stages.append(x.clone())

    return (x, stages) if return_stages else x


kernel, radius = make_gaussian_kernel(BASE_BLUR_SIGMA, device)
print(f"Base operator: Gaussian sigma={BASE_BLUR_SIGMA}, radius={radius}, "
      f"taps={2 * radius + 1}")

gate_rows, detail_rows = [], []

for image_path in tqdm(image_paths, desc="Exp 5.0 gate"):

    clean = load_rgb_image(image_path)
    with torch.no_grad():
        _, _, curves = model(clean)

    vanilla = apply_zero_dce_curves(clean, curves)
    modified = gamma_recurrence(clean, curves, 1.0, kernel, radius)
    error = (modified - vanilla).abs()

    gate_rows.append({
        "image": image_path.name,
        "max_abs_error": float(error.max().item()),
        "mean_abs_error": float(error.mean().item()),
        "vanilla_mean_luminance": float(vanilla.mean().item()),
    })

    base = gaussian_base(clean, kernel, radius)
    detail = clean - base
    luminance = (0.299 * clean[:, 0:1] + 0.587 * clean[:, 1:2] + 0.114 * clean[:, 2:3])
    detail_energy = (detail ** 2).mean(dim=1, keepdim=True)
    total_energy = (clean ** 2).mean(dim=1, keepdim=True)

    for region, (lower, upper) in BRIGHTNESS_BINS.items():
        mask = (luminance >= lower) & (luminance < upper)
        if mask.any():
            detail_rows.append({
                "image": image_path.name,
                "region": region,
                "pixel_count": int(mask.sum().item()),
                "mean_detail_energy": float(detail_energy[mask].mean().item()),
                "detail_fraction_of_total": float(
                    detail_energy[mask].mean().item()
                    / max(total_energy[mask].mean().item(), 1e-12)),
            })

gate_df = pd.DataFrame(gate_rows)
detail_df = pd.DataFrame(detail_rows)
gate_df.to_csv(EXP5_DIR / "experiment_5_0_gate.csv", index=False)
detail_df.to_csv(EXP5_DIR / "experiment_5_0_detail_energy.csv", index=False)

worst = gate_df["max_abs_error"].max()
print("\nGATE - gamma = 1 vs vanilla Zero-DCE")
print(gate_df[["image", "max_abs_error", "mean_abs_error"]].round(10).to_string(index=False))
print(f"\nWorst max_abs_error: {worst:.3e}")
print("GATE PASSED" if worst < 1e-5 else "GATE FAILED - stop and investigate")

print("\nDetail-layer energy by brightness region")
print(detail_df.groupby("region")[
    ["mean_detail_energy", "detail_fraction_of_total", "pixel_count"]
].mean().round(8).to_string())

## Experiment 5.1 - gamma sweep, no injected noise

**Run only if 5.0 passed.**

The question: does cutting detail gain cost us brightness?

- `mean_luminance` should stay roughly **flat** across gamma. That is the core claim.
- If it tracks gamma, the base is leaking brightness - raise `BASE_BLUR_SIGMA` and retry.
- `ssim_vs_vanilla` **will** fall as gamma drops. That is the honest trade: a global
  gamma suppresses real texture along with noise. How fast it falls tells us how much
  headroom a learned per-pixel gamma has.

In [ ]:
# ============================================================
# EXPERIMENT 5.1 - gamma sweep, no injected noise
# Question: does reducing detail gain cost us brightness?
# ============================================================

from skimage.metrics import structural_similarity as ssim_metric
from skimage.metrics import peak_signal_noise_ratio as psnr_metric


def to_numpy_image(tensor):
    return tensor.detach().squeeze(0).permute(1, 2, 0).cpu().numpy()


rows = []

for image_path in tqdm(image_paths, desc="Exp 5.1 sweep"):

    clean = load_rgb_image(image_path)
    with torch.no_grad():
        _, _, curves = model(clean)

    vanilla = gamma_recurrence(clean, curves, 1.0, kernel, radius)
    vanilla_np = np.clip(to_numpy_image(vanilla), 0.0, 1.0)

    reference_np = None
    reference_path = HIGH_DIR / image_path.name
    if reference_path.exists():
        reference_np = to_numpy_image(load_rgb_image(reference_path))

    for gamma in GAMMAS:

        output = gamma_recurrence(clean, curves, gamma, kernel, radius)
        output_np = np.clip(to_numpy_image(output), 0.0, 1.0)

        luminance_out = (0.299 * output[:, 0:1]
                         + 0.587 * output[:, 1:2]
                         + 0.114 * output[:, 2:3])

        row = {
            "image": image_path.name,
            "gamma": gamma,
            "mean_luminance": float(luminance_out.mean().item()),
            "fraction_above_one": float((output > 1.0).float().mean().item()),
            "fraction_below_zero": float((output < 0.0).float().mean().item()),
            "ssim_vs_vanilla": float(
                ssim_metric(vanilla_np, output_np, channel_axis=2, data_range=1.0)),
        }

        if reference_np is not None:
            row["psnr_vs_reference"] = float(
                psnr_metric(reference_np, output_np, data_range=1.0))
            row["ssim_vs_reference"] = float(
                ssim_metric(reference_np, output_np, channel_axis=2, data_range=1.0))

        rows.append(row)

sweep_df = pd.DataFrame(rows)
sweep_df.to_csv(EXP5_DIR / "experiment_5_1_per_image.csv", index=False)

summary_5_1 = sweep_df.groupby("gamma").mean(numeric_only=True).reset_index()
summary_5_1.to_csv(EXP5_DIR / "experiment_5_1_summary.csv", index=False)

print("\nEXPERIMENT 5.1 - gamma sweep, no injected noise")
print(summary_5_1.round(5).to_string(index=False))

## Experiment 5.2 - noise amplification under the gamma recurrence

**Consistency check first:** at `gamma = 1` this must recover Phase A's
78.35 / 82.65 / 104.22. Statistically, not bit-for-bit - `BASE_SEED` was
reconstructed, not recovered.

At `gamma = 0`, detail gain is exactly 1, so amplification should land near **1x**
at the same brightness.

In [ ]:
# ============================================================
# EXPERIMENT 5.2 - noise amplification under the gamma recurrence
# Frozen clean-image curves (branch 2A protocol from Phase A Exp 3).
#
# Checkpoint: at gamma = 1 this must recover the Phase A numbers
#             78.35 / 82.65 / 104.22 for sigma 0.005 / 0.010 / 0.020.
# ============================================================

rows = []

for image_index, image_path in enumerate(tqdm(image_paths, desc="Exp 5.2")):

    clean = load_rgb_image(image_path)
    with torch.no_grad():
        _, _, curves = model(clean)

    luminance = (0.299 * clean[:, 0:1] + 0.587 * clean[:, 1:2] + 0.114 * clean[:, 2:3])
    region_masks = {r: ((luminance >= lo) & (luminance < hi))
                    for r, (lo, hi) in BRIGHTNESS_BINS.items()}

    for sigma in SIGMAS:
        for gamma in GAMMAS:

            # Reference trajectory: same gamma, no noise.
            # Accumulating residuals avoids catastrophic cancellation.
            reference = gamma_recurrence(clean, curves, gamma, kernel, radius).double()

            out_sum = torch.zeros_like(reference)
            out_sum_sq = torch.zeros_like(reference)
            in_sum = torch.zeros_like(reference)
            in_sum_sq = torch.zeros_like(reference)

            generator = torch.Generator(device="cpu").manual_seed(
                EXP5_SEED + 1000 * image_index + int(sigma * 100000))

            for trial in range(N_TRIALS):

                noise = torch.randn(clean.shape, generator=generator,
                                    dtype=torch.float32).to(device) * sigma
                noisy = clean + noise

                output = gamma_recurrence(noisy, curves, gamma, kernel, radius).double()

                out_residual = output - reference
                in_residual = noisy.double() - clean.double()

                out_sum += out_residual
                out_sum_sq += out_residual ** 2
                in_sum += in_residual
                in_sum_sq += in_residual ** 2

            n = N_TRIALS
            out_var = torch.clamp((out_sum_sq - out_sum ** 2 / n) / (n - 1), min=0.0)
            in_var = torch.clamp((in_sum_sq - in_sum ** 2 / n) / (n - 1), min=0.0)
            mean_in_var = float(in_var.mean().item())

            row = {
                "image": image_path.name,
                "sigma": sigma,
                "gamma": gamma,
                "input_variance": mean_in_var,
                "output_variance": float(out_var.mean().item()),
                "variance_amplification": float(out_var.mean().item()) / max(mean_in_var, 1e-18),
            }

            out_var_luma = out_var.mean(dim=1, keepdim=True)
            for region, mask in region_masks.items():
                row[f"amp_{region}"] = (
                    float(out_var_luma[mask].mean().item()) / max(mean_in_var, 1e-18)
                    if mask.any() else np.nan)

            rows.append(row)

exp5_2_df = pd.DataFrame(rows)
exp5_2_df.to_csv(EXP5_DIR / "experiment_5_2_per_image.csv", index=False)

summary_5_2 = exp5_2_df.groupby(["sigma", "gamma"]).mean(numeric_only=True).reset_index()
summary_5_2.to_csv(EXP5_DIR / "experiment_5_2_summary.csv", index=False)

print("\nEXPERIMENT 5.2 - noise amplification vs gamma")
print(summary_5_2[["sigma", "gamma", "variance_amplification",
                   "amp_dark", "amp_mid_dark"]].round(4).to_string(index=False))

## Experiment 5.3 - the decoupling ratio

This is the figure the paper is built on.

Two perturbations of identical variance: white noise (lives in the detail layer)
and a smooth low-frequency perturbation (lives in the base). Measure the response
to each.

| result | reading |
|---|---|
| R ~ 1.0 at gamma = 1 | control arm correct - pointwise methods sit at exactly 1 |
| R > 1 at gamma < 1 | decoupling is real; this is the contribution |
| R ~ 1.0 at **every** gamma | decomposition is not separating structure from noise - switch to a guided filter before concluding anything |

In [ ]:
# ============================================================
# EXPERIMENT 5.3 - decoupling ratio
#
# Same variance, two perturbation types:
#   unstructured (white noise)  -> lives in the detail layer
#   structured  (low frequency) -> lives in the base layer
#
# R = amplification(structured) / amplification(unstructured)
#
# Any pointwise curve method - vanilla Zero-DCE, ZLEN, any curve
# reformulation - has R = 1 identically. R > 1 is the contribution.
# ============================================================

STRUCT_BLUR_SIGMA = 4.0
EXP53_TRIALS = 20
EXP53_SIGMA = 0.010

struct_kernel, struct_radius = make_gaussian_kernel(STRUCT_BLUR_SIGMA, device)


def make_perturbation(shape, sigma, kind, generator):
    noise = torch.randn(shape, generator=generator, dtype=torch.float32).to(device)
    if kind == "unstructured":
        return noise * sigma
    smooth = gaussian_base(noise, struct_kernel, struct_radius)
    # renormalise so both perturbation types carry the same variance
    return smooth * (sigma / max(float(smooth.std().item()), 1e-12))


rows = []

for image_index, image_path in enumerate(tqdm(image_paths, desc="Exp 5.3")):

    clean = load_rgb_image(image_path)
    with torch.no_grad():
        _, _, curves = model(clean)

    for gamma in GAMMAS:

        reference = gamma_recurrence(clean, curves, gamma, kernel, radius).double()
        record = {"image": image_path.name, "gamma": gamma}

        for kind in ["unstructured", "structured"]:

            out_sum = torch.zeros_like(reference)
            out_sum_sq = torch.zeros_like(reference)
            in_sum = torch.zeros_like(reference)
            in_sum_sq = torch.zeros_like(reference)

            generator = torch.Generator(device="cpu").manual_seed(
                EXP5_SEED + 7919 * image_index + (0 if kind == "unstructured" else 13))

            for trial in range(EXP53_TRIALS):

                perturbation = make_perturbation(clean.shape, EXP53_SIGMA, kind, generator)
                output = gamma_recurrence(clean + perturbation, curves,
                                          gamma, kernel, radius).double()

                out_residual = output - reference
                in_residual = perturbation.double()

                out_sum += out_residual
                out_sum_sq += out_residual ** 2
                in_sum += in_residual
                in_sum_sq += in_residual ** 2

            n = EXP53_TRIALS
            out_var = torch.clamp((out_sum_sq - out_sum ** 2 / n) / (n - 1), min=0.0)
            in_var = torch.clamp((in_sum_sq - in_sum ** 2 / n) / (n - 1), min=0.0)

            record[f"amp_{kind}"] = (float(out_var.mean().item())
                                     / max(float(in_var.mean().item()), 1e-18))

        record["decoupling_ratio"] = (record["amp_structured"]
                                      / max(record["amp_unstructured"], 1e-18))
        rows.append(record)

exp5_3_df = pd.DataFrame(rows)
exp5_3_df.to_csv(EXP5_DIR / "experiment_5_3_per_image.csv", index=False)

summary_5_3 = exp5_3_df.groupby("gamma").mean(numeric_only=True).reset_index()
summary_5_3.to_csv(EXP5_DIR / "experiment_5_3_summary.csv", index=False)

print("\nEXPERIMENT 5.3 - decoupling ratio")
print(summary_5_3.round(4).to_string(index=False))
print("\nAt gamma = 1 (vanilla Zero-DCE) the ratio should sit near 1.0.")
print("Any value meaningfully above 1.0 is decoupling no pointwise curve can achieve.")

---
## After 5.0-5.3

1. **Literature check before committing.** Base/detail decomposition with separate
   gain control is textbook tone mapping (unsharp masking, Durand & Dorsey 2002,
   local Laplacian filters). The novelty must rest on the impossibility result for
   pointwise curve LLIE, the corrected account of ZLEN's mechanism, the iterative
   zero-reference setting, and doing it with no ground truth. Check the Retinex
   decomposition line (RetinexNet, KinD, URetinex) and self-supervised LLIE
   (SCI, RUAS, PairLIE).

2. **Make gamma a learned per-pixel map.** The crux is separating texture from
   noise. Use structure-tensor **coherence**, not gradient magnitude - magnitude is
   large for edges *and* noise and cannot tell them apart. That was the flaw in the
   old `L_noise` flatness mask.

3. **Guard against the smoothing cheat.** The old objective let the model dim the
   image; this one lets it smooth the image. The loss needs a structure-preservation
   term.

4. **Watch for halos.** Brightening a blurred base while suppressing detail is the
   classic halo recipe. An edge-aware base (guided filter, O(N) via box filters)
   keeps edges in the base where they belong.